<!-- NOTEBOOK_OVERVIEW -->
# 1. Ablation Study (M1/M2/M3 + IBVS Variants)

## 2. Introduction
This notebook is the canonical ablation runner for the dissertation. It isolates the contribution of lexical text features, handcrafted lexical flags, and IBVS variants under a fixed low-FPR evaluation protocol across ID validation/test data and multiple OOD sets.

## 3. Workflow Steps
1. Load the split-specific processed dataset and construct lexical, flag, and IBVS feature blocks.
2. Train the ablation ladder models: `M1`, `M2`, `M3 + IBVS v1`, `M3 + IBVS v2 TOTAL`, and `M3 + IBVS v2 STRUCTURED`.
3. Select deployment thresholds on `VAL` using the shared low-FPR threshold selector.
4. Evaluate each model on `VAL`, `TEST`, and all configured OOD sets.
5. Export canonical metrics and IBVS forensic diagnostics.

## 4. Evaluation and Protocol Notes
1. Two evaluation tracks are produced:
   - `ranking`: threshold-free score quality used for low-FPR operating-point claims.
   - `deployment_threshold`: fixed-threshold behavior used for operational interpretation.
2. Core metrics reported include:
   - `macro_f1`, `accuracy`
   - `ROC-AUC`, `AUC-PR`
   - `TPR@1% FPR`, `TPR@5% FPR`, `TPR@10% FPR`
3. Threshold selection is validation-only and uses `best_threshold_low_fpr_with_macro_guard(...)` with explicit metadata logging (`selection_mode`, achieved validation FPR, feasibility, etc.).
4. IBVS interpretability outputs include component breakdowns, activation summaries, false-positive cases, and false-positive trigger summaries.

## 5. Execution Notes
1. Set `SPLIT_TAG` explicitly (`A`, `B`, or `C`) before execution.
2. Use `WRITE_MINIMAL_OUTPUTS=0` when secondary OOD-specific metrics, slice tables, and forensic suffix outputs are required.
3. This notebook should run independently from a fresh kernel once the processed artifacts exist.


In [ ]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 1) Imports + config

import sys
from pathlib import Path
import re
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

from xgboost import XGBClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 2) Load processed dataset v2

import os

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Reload local modules to avoid stale notebook-kernel imports.
import importlib
import src.evaluation.eval_metrics as eval_metrics
import src.features.ibvs as ibvs_mod

importlib.reload(eval_metrics)
importlib.reload(ibvs_mod)

from src.evaluation.eval_metrics import (
    evaluate_predictions,
    results_to_dataframe,
    best_threshold_by_macro_f1,
    best_threshold_low_fpr_with_macro_guard,
    bootstrap_metric_ci,
    bootstrap_delta_ci,
    tpr_at_fpr,
)

from src.features.ibvs import (
    ibvs_v2,
    ibvs_v2_with_triggers,
    IBVS_V2_NUMERIC_COLUMNS,
)
from src.common.notebook_utils import (
    DEFAULT_OVERRIDE_PATTERNS,
    get_git_commit,
    make_flag_matrix,
    safe_qcut,
    text_stats,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# ============================
# DATASET SPLIT SELECTION
# Split A = original v2 (current baseline)
# Split B/C = repeatability resamples (same rows, new ID train/val/test, OOD fixed)
# ============================

SPLIT_TAG = os.getenv("SPLIT_TAG", "B").strip().upper()   # env override: A/B/C
WRITE_MINIMAL_OUTPUTS = os.getenv("WRITE_MINIMAL_OUTPUTS", "1").strip().lower() not in {"0", "false", "no"}
print("WRITE_MINIMAL_OUTPUTS:", WRITE_MINIMAL_OUTPUTS)

expected_filename_by_split = {
    "A": "jailbreak_benchmarks_processed_v2.csv",
    "B": "jailbreak_benchmarks_processed_v2_splitB.csv",
    "C": "jailbreak_benchmarks_processed_v2_splitC.csv",
}
if SPLIT_TAG not in expected_filename_by_split:
    raise ValueError("SPLIT_TAG must be 'A', 'B', or 'C'.")
processed_filename = expected_filename_by_split[SPLIT_TAG]

# Threshold policies
RANKING_TRACK_POLICY = "macro_f1"
THRESHOLD_POLICY = "low_fpr_enforced"  # deployment track: {"low_fpr_guarded", "low_fpr_enforced"}
TARGET_FPR = 0.05
MACRO_F1_TOLERANCE = 0.02

processed_path = DATA_PROCESSED / processed_filename
assert processed_path.name == expected_filename_by_split[SPLIT_TAG], "processed filename/split mismatch"

print("Using dataset:", processed_path)
print("Ranking track policy:", RANKING_TRACK_POLICY)
print("Deployment threshold policy:", THRESHOLD_POLICY)
print(f"  target_fpr={TARGET_FPR:.2f}, macro_f1_tolerance={MACRO_F1_TOLERANCE:.3f}")

df = pd.read_csv(processed_path)

split_counts = df["split"].value_counts()
required_splits = ["train", "val", "test", "ood_test"]
missing_splits = [s for s in required_splits if s not in split_counts.index]
if missing_splits:
    raise ValueError(f"Missing required split(s): {missing_splits}")

print("Rows:", len(df))
print("\nSplit counts:")
print(split_counts)
print("\nLabel counts:")
print(df["label"].value_counts())


GIT_COMMIT = get_git_commit(PROJECT_ROOT)
print("Git commit:", GIT_COMMIT)


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 2b) Duplicate config cell intentionally disabled
# Keep cell 2 as the single source of truth for split/policy configuration.
pass


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()

OOD_SPLIT_ORDER = ["ood_test", "ood_test_injection", "ood_test_injection_standard"]
df_ood_map = {}
for split_name in OOD_SPLIT_ORDER:
    d = df[df["split"] == split_name].copy()
    if not d.empty:
        df_ood_map[split_name] = d

if "ood_test" not in df_ood_map:
    raise ValueError("Missing required split 'ood_test'.")

df_ood = df_ood_map["ood_test"]
df_ood_secondary_map = {k: v for k, v in df_ood_map.items() if k != "ood_test"}
df_ood_injection = df_ood_map.get("ood_test_injection")
df_ood_injection_standard = df_ood_map.get("ood_test_injection_standard")

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)] + list(df_ood_map.items()):
    if d.empty:
        raise ValueError(f"Split '{name}' is empty.")
    print(f"{name:30s}", d.shape, d["label"].value_counts().to_dict())

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values
y_ood_map = {k: v["label"].values for k, v in df_ood_map.items()}
y_ood_secondary_map = {k: v for k, v in y_ood_map.items() if k != "ood_test"}
y_ood_injection = y_ood_map.get("ood_test_injection")
y_ood_injection_standard = y_ood_map.get("ood_test_injection_standard")

OOD_SECONDARY_SPLITS = [k for k in OOD_SPLIT_ORDER if k in df_ood_secondary_map]


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 3) TF–IDF features (fit train only)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

tfidf.fit(df_train["prompt_text"])

X_lex_train = tfidf.transform(df_train["prompt_text"])
X_lex_val   = tfidf.transform(df_val["prompt_text"])
X_lex_test  = tfidf.transform(df_test["prompt_text"])

X_lex_ood_map = {
    split_name: tfidf.transform(df_ood_map[split_name]["prompt_text"])
    for split_name in df_ood_map
}
X_lex_ood = X_lex_ood_map["ood_test"]
X_lex_ood_injection = X_lex_ood_map.get("ood_test_injection")

print()
print("TF–IDF shapes:", X_lex_train.shape, X_lex_val.shape, X_lex_test.shape, X_lex_ood.shape)
for ood_name, X in X_lex_ood_map.items():
    print(f"{ood_name:18s}", X.shape)


In [ ]:
# Cell Purpose: Construct lexical feature representations for model training and inference.
# 4) Lexical flags (shared helper implementation)

OVERRIDE_PATTERNS = DEFAULT_OVERRIDE_PATTERNS

X_flag_train = make_flag_matrix(df_train["prompt_text"], override_patterns=OVERRIDE_PATTERNS)
X_flag_val = make_flag_matrix(df_val["prompt_text"], override_patterns=OVERRIDE_PATTERNS)
X_flag_test = make_flag_matrix(df_test["prompt_text"], override_patterns=OVERRIDE_PATTERNS)

X_flag_ood_map = {
    split_name: make_flag_matrix(df_ood_map[split_name]["prompt_text"], override_patterns=OVERRIDE_PATTERNS)
    for split_name in df_ood_map
}
X_flag_ood = X_flag_ood_map["ood_test"]
X_flag_ood_injection = X_flag_ood_map.get("ood_test_injection")

print()
print("Flag shapes:", X_flag_train.shape, X_flag_val.shape, X_flag_test.shape, X_flag_ood.shape)
for ood_name, X in X_flag_ood_map.items():
    print(f"{ood_name:18s}", X.shape)


In [ ]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 5) IBVS v1 (Legacy) Implementation

STRUCTURAL_PATTERNS_V1 = {
    "role_redefine": [
        "you are now",
        "from now on you will",
        "pretend to be",
        "act as",
    ],
    "ignore_boundary": [
        "ignore the above",
        "ignore previous instructions",
        "disregard all prior rules",
    ],
    "system_spoof": [
        "<system>",
        "[system]",
        "system:",
    ],
    "meta_override": [
        "as the system prompt",
        "override all policies",
    ],
}

IMPERATIVE_VERBS_V1 = [
    "write", "create", "develop", "design", "explain",
    "generate", "produce", "summarise", "summarize",
    "give", "show", "provide", "list", "teach"
]

HARMFUL_ACTION_VERBS_V1 = [
    "bypass", "exploit", "manufacture", "build",
    "construct", "smuggle", "hack", "steal",
    "traffic", "fabricate", "weapon", "poison"
]

INSTRUCTION_PATTERNS_V1 = [
    r"step[- ]?by[- ]?step",
    r"\bfirst\b",
    r"\bnext\b",
    r"\bthen\b",
    r"\bfinally\b"
]


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# Purpose: 04_ablation_studies.ipynb
def ibvs_v1_with_breakdown(text: str) -> tuple[float, dict]:
    """
    Instruction Boundary Violation Score v1
    - Structural overrides (role redefinition, ignoring boundaries, system spoofing, meta override)
    - Imperative starts
    - Harmful action verbs
    - Multi-step instructional structure
    - Returns (score, breakdown_dict)
    - Score is v1's token length-normalised score
    """
    if not isinstance(text, str):
        return 0.0, {
            "role_redefine": 0.0,
            "ignore_boundary": 0.0,
            "system_spoof": 0.0,
            "meta_override": 0.0,
            "imperative_start": 0.0,
            "harmful_verbs": 0.0,
            "instructional_structure": 0.0,
            "instructional_metawords": 0.0,
            "length_tokens": 0.0,
        }

    t = text.lower()
    score = 0.0

    bd = {
        "role_redefine": 0.0,
        "ignore_boundary": 0.0,
        "system_spoof": 0.0,
        "meta_override": 0.0,
        "imperative_start": 0.0,
        "harmful_verbs": 0.0,
        "instructional_structure": 0.0,
        "instructional_metawords": 0.0,
        "length_tokens": 0.0,
    }

    # Explicit structural overrides
    if any(p in t for p in STRUCTURAL_PATTERNS_V1["role_redefine"]):
        score += 2.0
        bd["role_redefine"] = 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS_V1["ignore_boundary"]):
        score += 2.0
        bd["ignore_boundary"] = 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS_V1["system_spoof"]):
        score += 2.0
        bd["system_spoof"] = 2.0
    if any(p in t for p in STRUCTURAL_PATTERNS_V1["meta_override"]):
        score += 2.0
        bd["meta_override"] = 2.0

    # Imperative first word
    tokens = t.split()
    first_word = tokens[0] if len(tokens) > 0 else ""
    if first_word in IMPERATIVE_VERBS_V1:
        score += 1.0
        bd["imperative_start"] = 1.0
    
    # Harmful action verbs
    if any(v in t for v in HARMFUL_ACTION_VERBS_V1):
        score += 1.5
        bd["harmful_verbs"] = 1.5

    # Multi-step cues
    if any(re.search(p, t) for p in INSTRUCTION_PATTERNS_V1):
        score += 1.0
        bd["instructional_structure"] = 1.0

    # Instructional meta-words
    for cue in ["guide", "instructions", "tutorial", "manual"]:
        if cue in t:
            score += 1.0
            bd["instructional_metawords"] = 1.0
            break

    length = max(len(tokens), 1)
    bd["length_tokens"] = float(length)

    # v1: strict length normalisation
    return float(score / length), bd


In [ ]:
# Cell Purpose: Compute IBVS structural features and associated trigger-level diagnostics.
# 6) IBVS v2 computation (total + breakdown + triggers)
def compute_ibvs_v2_scores(texts):
    """
    Returns:
      scores: np.ndarray (n,)
      breakdown_df: pd.DataFrame with numeric, interpretable sub-scores
      trigger_df: pd.DataFrame with trigger metadata for explainability
    """
    scores = []
    rows = []
    trigger_rows = []
    for t in texts:
        s, breakdown, triggers = ibvs_v2_with_triggers(t)
        scores.append(s)
        rows.append(breakdown)
        trigger_rows.append(
            {
                "trigger_count": len(triggers),
                "triggered_rules": "|".join(triggers),
            }
        )

    breakdown_df = pd.DataFrame(rows)
    for c in IBVS_V2_NUMERIC_COLUMNS:
        if c not in breakdown_df.columns:
            breakdown_df[c] = 0.0
    breakdown_df = breakdown_df[list(IBVS_V2_NUMERIC_COLUMNS)].astype(float)

    trigger_df = pd.DataFrame(trigger_rows)
    return np.array(scores, dtype=float), breakdown_df, trigger_df


def compute_ibvs_v1_scores(texts):
    """
    Returns:
      scores: np.ndarray (n,)
      breakdown_df: pd.DataFrame with v1 breakdown
    """
    scores = []
    rows = []
    for t in texts:
        s, breakdown = ibvs_v1_with_breakdown(t)
        scores.append(s)
        rows.append(breakdown)
    return np.array(scores, dtype=float), pd.DataFrame(rows)


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# compute for each split
train_texts = df_train["prompt_text"].astype(str).tolist()
val_texts   = df_val["prompt_text"].astype(str).tolist()
test_texts  = df_test["prompt_text"].astype(str).tolist()

ood_texts_map = {
    split_name: df_ood_map[split_name]["prompt_text"].astype(str).tolist()
    for split_name in df_ood_map
}
ood_texts = ood_texts_map["ood_test"]

ibvs1_train, ibvs1_bd_train = compute_ibvs_v1_scores(train_texts)
ibvs1_val,   ibvs1_bd_val   = compute_ibvs_v1_scores(val_texts)
ibvs1_test,  ibvs1_bd_test  = compute_ibvs_v1_scores(test_texts)

ibvs2_train, ibvs2_bd_train, ibvs2_tr_train = compute_ibvs_v2_scores(train_texts)
ibvs2_val,   ibvs2_bd_val,   ibvs2_tr_val   = compute_ibvs_v2_scores(val_texts)
ibvs2_test,  ibvs2_bd_test,  ibvs2_tr_test  = compute_ibvs_v2_scores(test_texts)

ibvs1_ood_map = {}
ibvs1_bd_ood_map = {}
ibvs2_ood_map = {}
ibvs2_bd_ood_map = {}
ibvs2_tr_ood_map = {}

for split_name, txts in ood_texts_map.items():
    s1, bd1 = compute_ibvs_v1_scores(txts)
    s2, bd2, tr2 = compute_ibvs_v2_scores(txts)
    ibvs1_ood_map[split_name] = s1
    ibvs1_bd_ood_map[split_name] = bd1
    ibvs2_ood_map[split_name] = s2
    ibvs2_bd_ood_map[split_name] = bd2
    ibvs2_tr_ood_map[split_name] = tr2

ibvs1_ood = ibvs1_ood_map["ood_test"]
ibvs1_bd_ood = ibvs1_bd_ood_map["ood_test"]
ibvs2_ood = ibvs2_ood_map["ood_test"]
ibvs2_bd_ood = ibvs2_bd_ood_map["ood_test"]
ibvs2_tr_ood = ibvs2_tr_ood_map["ood_test"]

ibvs1_ood_injection = ibvs1_ood_map.get("ood_test_injection")
ibvs1_bd_ood_injection = ibvs1_bd_ood_map.get("ood_test_injection")
ibvs2_ood_injection = ibvs2_ood_map.get("ood_test_injection")
ibvs2_bd_ood_injection = ibvs2_bd_ood_map.get("ood_test_injection")
ibvs2_tr_ood_injection = ibvs2_tr_ood_map.get("ood_test_injection")

print()
print("IBVS v1 totals:", ibvs1_train.shape, ibvs1_val.shape, ibvs1_test.shape, ibvs1_ood.shape)
print("IBVS v2 totals:", ibvs2_train.shape, ibvs2_val.shape, ibvs2_test.shape, ibvs2_ood.shape)
for split_name, vals in ibvs2_ood_map.items():
    print(f"{split_name:18s} IBVS v2 shape: {vals.shape}")
print("IBVS v2 numeric columns:", list(IBVS_V2_NUMERIC_COLUMNS))


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 7) Build feature matrices for each model
#   - M1: TF-IDF
#   - M2: TF-IDF + flags
#   - M3_V1_TOTAL: TF-IDF + flags + ibvs_v1_total
#   - M3_V2_TOTAL: TF-IDF + flags + ibvs_v2_total
#   - M3_V2_STRUCTURED: TF-IDF + flags + ibvs_v2 structured components

def colvec(x: np.ndarray) -> csr_matrix:
    return csr_matrix(x.reshape(-1, 1).astype(float))


def dense_df_to_csr(df_block: pd.DataFrame) -> csr_matrix:
    return csr_matrix(df_block.values.astype(float))


X_m1_train, X_m1_val, X_m1_test, X_m1_ood = X_lex_train, X_lex_val, X_lex_test, X_lex_ood

X_m2_train = hstack([X_lex_train, X_flag_train]).tocsr()
X_m2_val   = hstack([X_lex_val,   X_flag_val]).tocsr()
X_m2_test  = hstack([X_lex_test,  X_flag_test]).tocsr()
X_m2_ood   = hstack([X_lex_ood,   X_flag_ood]).tocsr()

# Scalar IBVS variants (legacy comparators)
X_m3_v1_train = hstack([X_lex_train, X_flag_train, colvec(ibvs1_train)]).tocsr()
X_m3_v1_val   = hstack([X_lex_val,   X_flag_val,   colvec(ibvs1_val)]).tocsr()
X_m3_v1_test  = hstack([X_lex_test,  X_flag_test,  colvec(ibvs1_test)]).tocsr()
X_m3_v1_ood   = hstack([X_lex_ood,   X_flag_ood,   colvec(ibvs1_ood)]).tocsr()

X_m3_v2_total_train = hstack([X_lex_train, X_flag_train, colvec(ibvs2_train)]).tocsr()
X_m3_v2_total_val   = hstack([X_lex_val,   X_flag_val,   colvec(ibvs2_val)]).tocsr()
X_m3_v2_total_test  = hstack([X_lex_test,  X_flag_test,  colvec(ibvs2_test)]).tocsr()
X_m3_v2_total_ood   = hstack([X_lex_ood,   X_flag_ood,   colvec(ibvs2_ood)]).tocsr()

# Structured IBVS v2 components.
ibvs2_feat_train = ibvs2_bd_train.copy()
ibvs2_feat_val   = ibvs2_bd_val.copy()
ibvs2_feat_test  = ibvs2_bd_test.copy()
ibvs2_feat_ood   = ibvs2_bd_ood.copy()

ibvs2_feat_train["ibvs_v2_total"] = ibvs2_train
ibvs2_feat_val["ibvs_v2_total"]   = ibvs2_val
ibvs2_feat_test["ibvs_v2_total"]  = ibvs2_test
ibvs2_feat_ood["ibvs_v2_total"]   = ibvs2_ood

X_ibvs2_struct_train = dense_df_to_csr(ibvs2_feat_train)
X_ibvs2_struct_val   = dense_df_to_csr(ibvs2_feat_val)
X_ibvs2_struct_test  = dense_df_to_csr(ibvs2_feat_test)
X_ibvs2_struct_ood   = dense_df_to_csr(ibvs2_feat_ood)

X_m3_v2_struct_train = hstack([X_lex_train, X_flag_train, X_ibvs2_struct_train]).tocsr()
X_m3_v2_struct_val   = hstack([X_lex_val,   X_flag_val,   X_ibvs2_struct_val]).tocsr()
X_m3_v2_struct_test  = hstack([X_lex_test,  X_flag_test,  X_ibvs2_struct_test]).tocsr()
X_m3_v2_struct_ood   = hstack([X_lex_ood,   X_flag_ood,   X_ibvs2_struct_ood]).tocsr()

print("
Feature shapes:")
print("M1:", X_m1_train.shape)
print("M2:", X_m2_train.shape)
print("M3_V1_TOTAL:", X_m3_v1_train.shape)
print("M3_V2_TOTAL:", X_m3_v2_total_train.shape)
print("M3_V2_STRUCTURED:", X_m3_v2_struct_train.shape)

# Secondary OOD feature maps (evaluation only; no retraining).
OOD_FEATURES = {
    "M1_TFIDF_ONLY": {},
    "M2_TFIDF_PLUS_FLAGS": {},
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL": {},
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL": {},
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED": {},
}

for ood_name in OOD_SECONDARY_SPLITS:
    X_lex_o = X_lex_ood_map[ood_name]
    X_flag_o = X_flag_ood_map[ood_name]
    ibvs1_o = ibvs1_ood_map[ood_name]
    ibvs2_o = ibvs2_ood_map[ood_name]

    ibvs2_feat_o = ibvs2_bd_ood_map[ood_name].copy()
    ibvs2_feat_o["ibvs_v2_total"] = ibvs2_o

    X_ibvs2_struct_o = dense_df_to_csr(ibvs2_feat_o)

    OOD_FEATURES["M1_TFIDF_ONLY"][ood_name] = X_lex_o
    OOD_FEATURES["M2_TFIDF_PLUS_FLAGS"][ood_name] = hstack([X_lex_o, X_flag_o]).tocsr()
    OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL"][ood_name] = hstack([X_lex_o, X_flag_o, colvec(ibvs1_o)]).tocsr()
    OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL"][ood_name] = hstack([X_lex_o, X_flag_o, colvec(ibvs2_o)]).tocsr()
    OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED"][ood_name] = hstack([X_lex_o, X_flag_o, X_ibvs2_struct_o]).tocsr()

    print(f"Secondary OOD feature shapes ({ood_name}):")
    for model_name in OOD_FEATURES:
        print(f" - {model_name}: {OOD_FEATURES[model_name][ood_name].shape}")


In [ ]:
# Cell Purpose: Train model(s) using the prepared feature sets and split configuration.
# 8) Train helper function
def train_xgb(X_train, y_train) -> XGBClassifier:
    """
    Train XGBoost with fixed hyperparams across ablations for fairness.
    """
    xgb = XGBClassifier(
        objective="binary:logistic",
        n_estimators=400,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=2,
        reg_lambda=2.0,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    xgb.fit(X_train, y_train, verbose=False)
    return xgb


In [ ]:
# Cell Purpose: Train model(s) using the prepared feature sets and split configuration.
# 9) Run Experiment: train once, evaluate both tracks

MODEL_ARTIFACTS = []


def _select_threshold(y_val, val_proba, policy_name: str):
    if policy_name == "macro_f1":
        t_star, best_val_f1 = best_threshold_by_macro_f1(y_val, val_proba, n_grid=1001)
        meta = {
            "selection_mode": "macro_f1",
            "target_fpr": np.nan,
            "macro_f1_tolerance": np.nan,
            "enforce_target_fpr": False,
            "fallback_to_macro_f1": False,
            "val_macro_f1_best": float(best_val_f1),
            "val_macro_f1_at_t_star": float(best_val_f1),
            "val_fpr_at_t_star": np.nan,
            "val_tpr_at_t_star": np.nan,
            "val_fpr_constraint_satisfied": np.nan,
            "val_fpr_gap_to_target": np.nan,
            "val_fpr_min_possible": np.nan,
            "val_num_feasible_thresholds": np.nan,
        }
        return float(t_star), meta

    if policy_name in {"low_fpr_guarded", "low_fpr_enforced"}:
        t_star, meta = best_threshold_low_fpr_with_macro_guard(
            y_val,
            val_proba,
            target_fpr=TARGET_FPR,
            macro_f1_tolerance=MACRO_F1_TOLERANCE,
            enforce_target_fpr=(policy_name == "low_fpr_enforced"),
            fallback_to_macro_f1=True,
            n_grid=1001,
        )
        return float(t_star), meta

    raise ValueError("Unknown policy_name")


def _evaluate_track(model_name, eval_track, policy_name, y_val, y_test, y_ood, val_proba, test_proba, ood_proba):
    t_star, threshold_meta = _select_threshold(y_val, val_proba, policy_name)

    val_pred = (val_proba >= t_star).astype(int)
    test_pred = (test_proba >= t_star).astype(int)
    ood_pred = (ood_proba >= t_star).astype(int)

    note = (
        f"eval_track={eval_track}; policy={policy_name}; mode={threshold_meta.get('selection_mode')}; "
        f"t*={t_star:.3f}; target_fpr={TARGET_FPR:.2f}; "
        f"val_fpr={threshold_meta.get('val_fpr_at_t_star', np.nan):.3f}; macro_tol={MACRO_F1_TOLERANCE:.3f}"
    )

    val_res = evaluate_predictions("VAL", y_val, val_pred, val_proba, print_report=True, threshold_note=note)
    test_res = evaluate_predictions("TEST", y_test, test_pred, test_proba, print_report=True, threshold_note=note)
    ood_res = evaluate_predictions("OOD", y_ood, ood_pred, ood_proba, print_report=True, threshold_note=note)

    df_out = results_to_dataframe(model_name, [val_res, test_res, ood_res])
    df_out["eval_track"] = eval_track
    df_out["threshold_policy"] = policy_name
    df_out["val_threshold_t_star"] = float(t_star)
    for k, v in threshold_meta.items():
        df_out[f"threshold_{k}"] = v

    artifact = {
        "model": model_name,
        "eval_track": eval_track,
        "policy_name": policy_name,
        "t_star": float(t_star),
        "threshold_meta": dict(threshold_meta),
        "val_proba": np.asarray(val_proba, dtype=float),
        "test_proba": np.asarray(test_proba, dtype=float),
        "ood_proba": np.asarray(ood_proba, dtype=float),
        "val_pred": np.asarray(val_pred, dtype=int),
        "test_pred": np.asarray(test_pred, dtype=int),
        "ood_pred": np.asarray(ood_pred, dtype=int),
    }

    return df_out, artifact


def run_experiment(model_name, X_train, X_val, X_test, X_ood, y_train, y_val, y_test, y_ood):
    clf = train_xgb(X_train, y_train)

    # Score vectors are threshold-free and shared across both evaluation tracks.
    val_proba = clf.predict_proba(X_val)[:, 1]
    test_proba = clf.predict_proba(X_test)[:, 1]
    ood_proba = clf.predict_proba(X_ood)[:, 1]

    ranking_df, ranking_artifact = _evaluate_track(
        model_name,
        eval_track="ranking",
        policy_name=RANKING_TRACK_POLICY,
        y_val=y_val,
        y_test=y_test,
        y_ood=y_ood,
        val_proba=val_proba,
        test_proba=test_proba,
        ood_proba=ood_proba,
    )

    deployment_df, deployment_artifact = _evaluate_track(
        model_name,
        eval_track="deployment_threshold",
        policy_name=THRESHOLD_POLICY,
        y_val=y_val,
        y_test=y_test,
        y_ood=y_ood,
        val_proba=val_proba,
        test_proba=test_proba,
        ood_proba=ood_proba,
    )

    for artifact in (ranking_artifact, deployment_artifact):
        artifact["y_val"] = np.asarray(y_val, dtype=int)
        artifact["y_test"] = np.asarray(y_test, dtype=int)
        artifact["y_ood"] = np.asarray(y_ood, dtype=int)
        MODEL_ARTIFACTS.append(artifact)

    return pd.concat([ranking_df, deployment_df], ignore_index=True)


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 10) Run Ablations
results_frames = []

results_frames.append(
    run_experiment("M1_TFIDF_ONLY", X_m1_train, X_m1_val, X_m1_test, X_m1_ood, y_train, y_val, y_test, y_ood)
)

results_frames.append(
    run_experiment("M2_TFIDF_PLUS_FLAGS", X_m2_train, X_m2_val, X_m2_test, X_m2_ood, y_train, y_val, y_test, y_ood)
)

results_frames.append(
    run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL",
        X_m3_v1_train, X_m3_v1_val, X_m3_v1_test, X_m3_v1_ood,
        y_train, y_val, y_test, y_ood,
    )
)

results_frames.append(
    run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL",
        X_m3_v2_total_train, X_m3_v2_total_val, X_m3_v2_total_test, X_m3_v2_total_ood,
        y_train, y_val, y_test, y_ood,
    )
)

results_frames.append(
    run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED",
        X_m3_v2_struct_train, X_m3_v2_struct_val, X_m3_v2_struct_test, X_m3_v2_struct_ood,
        y_train, y_val, y_test, y_ood,
    )
)

ablation_metrics = pd.concat(results_frames, ignore_index=True)

print()
print("=== Ablation Summary (dual-track eval protocol) ===")
display(ablation_metrics)


# Secondary OOD evaluation (same trained protocol; evaluation-only) written to suffixed outputs.
SECONDARY_MODEL_ARTIFACTS = {}
ablation_metrics_secondary = {}

for ood_name in OOD_SECONDARY_SPLITS:
    artifact_checkpoint = len(MODEL_ARTIFACTS)
    y_ood_curr = y_ood_map[ood_name]

    sec_frames = []
    sec_frames.append(run_experiment("M1_TFIDF_ONLY", X_m1_train, X_m1_val, X_m1_test, OOD_FEATURES["M1_TFIDF_ONLY"][ood_name], y_train, y_val, y_test, y_ood_curr))
    sec_frames.append(run_experiment("M2_TFIDF_PLUS_FLAGS", X_m2_train, X_m2_val, X_m2_test, OOD_FEATURES["M2_TFIDF_PLUS_FLAGS"][ood_name], y_train, y_val, y_test, y_ood_curr))
    sec_frames.append(run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL",
        X_m3_v1_train, X_m3_v1_val, X_m3_v1_test, OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL"][ood_name],
        y_train, y_val, y_test, y_ood_curr,
    ))
    sec_frames.append(run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL",
        X_m3_v2_total_train, X_m3_v2_total_val, X_m3_v2_total_test, OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL"][ood_name],
        y_train, y_val, y_test, y_ood_curr,
    ))
    sec_frames.append(run_experiment(
        "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED",
        X_m3_v2_struct_train, X_m3_v2_struct_val, X_m3_v2_struct_test, OOD_FEATURES["M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED"][ood_name],
        y_train, y_val, y_test, y_ood_curr,
    ))

    sec_df = pd.concat(sec_frames, ignore_index=True)
    sec_df = sec_df[sec_df["split"] == "OOD"].copy()
    sec_df["ood_name"] = ood_name

    ablation_metrics_secondary[ood_name] = sec_df
    SECONDARY_MODEL_ARTIFACTS[ood_name] = MODEL_ARTIFACTS[artifact_checkpoint:]
    MODEL_ARTIFACTS[:] = MODEL_ARTIFACTS[:artifact_checkpoint]

    print()
    print(f"=== Secondary OOD ({ood_name}) ablation summary ===")
    display(sec_df)


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 11) Save metrics (split-tagged, canonical output only)
OUT_METRICS_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
OUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

combined_path = OUT_METRICS_DIR / f"metrics_ablation_split{SPLIT_TAG}.csv"
if not combined_path.name.endswith(f"split{SPLIT_TAG}.csv"):
    raise ValueError("Output filename does not match active SPLIT_TAG.")

ablation_metrics.to_csv(combined_path, index=False)
print(f"\nSaved metrics (combined): {combined_path}")

if not WRITE_MINIMAL_OUTPUTS:
    for ood_name, df_sec in ablation_metrics_secondary.items():
        if df_sec is None or df_sec.empty:
            continue
        sec_path = OUT_METRICS_DIR / f"metrics_ablation_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_sec.to_csv(sec_path, index=False)
        print(f"Saved secondary OOD metrics ({ood_name}): {sec_path}")

    # Also write explicit primary OOD-suffixed metrics view.
    primary_ood_rows = ablation_metrics[ablation_metrics["split"] == "OOD"].copy()
    if not primary_ood_rows.empty:
        primary_ood_rows["ood_name"] = "ood_test"
        primary_suffix_path = OUT_METRICS_DIR / f"metrics_ablation_split{SPLIT_TAG}__ood-ood_test.csv"
        primary_ood_rows.to_csv(primary_suffix_path, index=False)
        print(f"Saved primary OOD-suffixed metrics: {primary_suffix_path}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped ablation OOD-suffixed metrics files.")


In [ ]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 12) Slice-level diagnostics (ID TEST + OOD), to compare IBVS activation and model metrics by slice
# Slices are defined from prompt text patterns + IBVS channels; OOD set itself remains unchanged.

OUT_IBVS_DIR = PROJECT_ROOT / "experiments" / "results" / "ibvs"
OUT_IBVS_DIR.mkdir(parents=True, exist_ok=True)

def _slice_masks(prompt_series: pd.Series, bd: pd.DataFrame):
    txt = prompt_series.astype(str).str.lower()

    boundary_regex = (
        r"ignore\b.{0,80}(previous|prior|above)|"
        r"disregard\b|override\b|bypass\b|"
        r"(system|developer)\s*:|<\s*(system|developer)\s*>|\[\s*(system|developer)\s*\]|"
        r"you are now|pretend to be|act as|from now on"
    )
    boundary_like = txt.str.contains(boundary_regex, regex=True, na=False)

    harm_like = (pd.to_numeric(bd.get("harm_domain", 0.0), errors="coerce").fillna(0.0) > 0.0)
    evasion_like = (pd.to_numeric(bd.get("evasion", 0.0), errors="coerce").fillna(0.0) > 0.0)
    harm_intent_like = harm_like | evasion_like

    return {
        "boundary_override_like": boundary_like.to_numpy(dtype=bool),
        "harm_intent_like": harm_intent_like.to_numpy(dtype=bool),
    }


def _activation_summary(stage_name: str, y: np.ndarray, bd: pd.DataFrame, masks: dict):
    rows = []
    high_cols = [
        "hierarchy_override",
        "system_spoof",
        "interaction_system_hierarchy_spoof_chain",
        "high_specific_risk_anchor",
        "tripwire_alert",
        "harm_domain",
        "evasion",
    ]
    for slice_name, pos_mask in masks.items():
        slice_pos = (y == 1) & pos_mask
        n_pos = int(slice_pos.sum())
        for col in high_cols:
            vals = pd.to_numeric(bd.get(col, 0.0), errors="coerce").fillna(0.0)
            rate = float((vals[slice_pos] > 0).mean()) if n_pos > 0 else np.nan
            rows.append(
                {
                    "stage": stage_name,
                    "slice_name": slice_name,
                    "n_slice_positive": n_pos,
                    "ibvs_component": col,
                    "activation_rate_on_slice_positive": rate,
                }
            )
    return rows


slice_rows = []
activation_rows = []

test_prompts = df_test["prompt_text"].reset_index(drop=True)
ood_prompts = df_ood["prompt_text"].reset_index(drop=True)

test_bd = ibvs2_bd_test.reset_index(drop=True)
ood_bd = ibvs2_bd_ood.reset_index(drop=True)

test_masks = _slice_masks(test_prompts, test_bd)
ood_masks = _slice_masks(ood_prompts, ood_bd)

action_specs = [
    ("TEST", np.asarray(y_test, dtype=int), test_masks),
    ("OOD", np.asarray(y_ood, dtype=int), ood_masks),
]

for stage_name, y_stage, masks in action_specs:
    bd_stage = test_bd if stage_name == "TEST" else ood_bd
    activation_rows.extend(_activation_summary(stage_name, y_stage, bd_stage, masks))

for artifact in MODEL_ARTIFACTS:
    model_name = artifact["model"]
    eval_track = artifact["eval_track"]

    for stage_name, y_stage, masks in action_specs:
        proba = np.asarray(artifact["test_proba" if stage_name == "TEST" else "ood_proba"], dtype=float)
        pred = np.asarray(artifact["test_pred" if stage_name == "TEST" else "ood_pred"], dtype=int)

        neg_mask = y_stage == 0
        n_neg_total = int(neg_mask.sum())

        for slice_name, pos_mask in masks.items():
            mask = neg_mask | ((y_stage == 1) & pos_mask)
            y_s = y_stage[mask]
            p_s = pred[mask]
            s_s = proba[mask]

            n_pos = int(((y_stage == 1) & pos_mask).sum())
            n_neg = int((y_stage[mask] == 0).sum())

            if len(y_s) == 0 or n_pos == 0 or n_neg == 0:
                continue

            note = (
                f"slice_eval={slice_name}; stage={stage_name}; eval_track={eval_track}; "
                f"n_pos={n_pos}; n_neg={n_neg}; n_neg_total_stage={n_neg_total}"
            )

            res = evaluate_predictions(
                split_name=f"{stage_name}_SLICE_{slice_name.upper()}",
                y_true=y_s,
                y_pred=p_s,
                y_score_for_metrics=s_s,
                print_report=False,
                threshold_note=note,
            )

            df_one = results_to_dataframe(model_name, [res])
            df_one["eval_track"] = eval_track
            df_one["slice_name"] = slice_name
            df_one["slice_stage"] = stage_name
            df_one["slice_positive_count"] = n_pos
            df_one["slice_negative_count"] = n_neg
            df_one["slice_negative_total_stage"] = n_neg_total
            df_one["slice_positive_prevalence_within_stage"] = float(n_pos / max(int((y_stage == 1).sum()), 1))
            df_one["split_tag"] = SPLIT_TAG
            slice_rows.append(df_one)

if slice_rows and (not WRITE_MINIMAL_OUTPUTS):
    df_slice_metrics = pd.concat(slice_rows, ignore_index=True)
    slice_metrics_path = OUT_METRICS_DIR / f"metrics_ablation_slice_split{SPLIT_TAG}.csv"
    df_slice_metrics.to_csv(slice_metrics_path, index=False)
    print(f"Saved slice metrics: {slice_metrics_path}")
    slice_primary_suffix = OUT_METRICS_DIR / f"metrics_ablation_slice_split{SPLIT_TAG}__ood-ood_test.csv"
    df_slice_primary = df_slice_metrics.copy()
    if "ood_name" not in df_slice_primary.columns:
        df_slice_primary["ood_name"] = "id"
    df_slice_primary.loc[df_slice_primary["slice_stage"].astype(str).eq("OOD"), "ood_name"] = "ood_test"
    df_slice_primary.to_csv(slice_primary_suffix, index=False)
    print(f"Saved primary OOD-suffixed slice metrics: {slice_primary_suffix}")
    display(df_slice_metrics.head(12))
else:
    print("No slice metrics were produced (likely zero positive slice membership).")

if activation_rows:
    df_activation = pd.DataFrame(activation_rows)
    activation_path = OUT_IBVS_DIR / f"ibvs_v2_activation_summary_split{SPLIT_TAG}.csv"
    df_activation.to_csv(activation_path, index=False)
    print(f"Saved IBVS activation summary: {activation_path}")
    if not WRITE_MINIMAL_OUTPUTS:
        activation_primary_suffix = OUT_IBVS_DIR / f"ibvs_v2_activation_summary_split{SPLIT_TAG}__ood-ood_test.csv"
        df_activation_primary = df_activation.copy()
        if "ood_name" not in df_activation_primary.columns:
            df_activation_primary["ood_name"] = "id"
        df_activation_primary.loc[df_activation_primary["stage"].astype(str).eq("OOD"), "ood_name"] = "ood_test"
        df_activation_primary.to_csv(activation_primary_suffix, index=False)
        print(f"Saved primary OOD-suffixed activation summary: {activation_primary_suffix}")
    display(df_activation.head(20))
else:
    print("No activation summary rows were produced.")


# Additional diagnostics: OOD/TEST performance by prompt-length and lexical-complexity bins
# Complexity proxy: type-token ratio (lexical diversity) on whitespace-tokenized text.

# Bin construction reuses shared text/quantile helpers from src.common.notebook_utils.

bin_rows = []
for stage_name, y_stage, _ in action_specs:
    prompt_stage = test_prompts if stage_name == "TEST" else ood_prompts
    stats_stage = text_stats(prompt_stage)
    stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
    stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

    for artifact in MODEL_ARTIFACTS:
        model_name = artifact["model"]
        eval_track = artifact["eval_track"]

        proba = np.asarray(artifact["test_proba" if stage_name == "TEST" else "ood_proba"], dtype=float)
        pred = np.asarray(artifact["test_pred" if stage_name == "TEST" else "ood_pred"], dtype=int)

        for bin_family in ["length_bin", "complexity_bin"]:
            for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
                mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
                y_s = np.asarray(y_stage, dtype=int)[mask]
                p_s = pred[mask]
                s_s = proba[mask]

                n_total = int(mask.sum())
                n_pos = int((y_s == 1).sum())
                n_neg = int((y_s == 0).sum())

                if n_total == 0 or n_pos == 0 or n_neg == 0:
                    continue

                note = (
                    f"bin_eval={bin_family}:{bin_label}; stage={stage_name}; eval_track={eval_track}; "
                    f"n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}"
                )

                res = evaluate_predictions(
                    split_name=f"{stage_name}_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                    y_true=y_s,
                    y_pred=p_s,
                    y_score_for_metrics=s_s,
                    print_report=False,
                    threshold_note=note,
                )

                # For harmful prompts, evasion is a false negative (predicted benign).
                evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))

                df_one = results_to_dataframe(model_name, [res])
                df_one["eval_track"] = eval_track
                df_one["slice_stage"] = stage_name
                df_one["bin_family"] = bin_family
                df_one["bin_label"] = str(bin_label)
                df_one["bin_count"] = n_total
                df_one["bin_positive_count"] = n_pos
                df_one["bin_negative_count"] = n_neg
                df_one["evasion_rate"] = evasion_rate
                df_one["token_count_median"] = float(np.median(stats_stage.loc[mask, "token_count"].astype(float)))
                df_one["token_count_mean"] = float(np.mean(stats_stage.loc[mask, "token_count"].astype(float)))
                df_one["lexical_ttr_median"] = float(np.median(stats_stage.loc[mask, "lexical_ttr"].astype(float)))
                df_one["avg_token_len_median"] = float(np.median(stats_stage.loc[mask, "avg_token_len"].astype(float)))
                df_one["split_tag"] = SPLIT_TAG
                bin_rows.append(df_one)

if bin_rows and (not WRITE_MINIMAL_OUTPUTS):
    df_bin_metrics = pd.concat(bin_rows, ignore_index=True)
    bin_metrics_path = OUT_METRICS_DIR / f"metrics_ablation_bins_split{SPLIT_TAG}.csv"
    df_bin_metrics.to_csv(bin_metrics_path, index=False)
    print(f"Saved length/complexity bin metrics: {bin_metrics_path}")
    bin_primary_suffix = OUT_METRICS_DIR / f"metrics_ablation_bins_split{SPLIT_TAG}__ood-ood_test.csv"
    df_bin_primary = df_bin_metrics.copy()
    if "ood_name" not in df_bin_primary.columns:
        df_bin_primary["ood_name"] = "id"
    df_bin_primary.loc[df_bin_primary["slice_stage"].astype(str).eq("OOD"), "ood_name"] = "ood_test"
    df_bin_primary.to_csv(bin_primary_suffix, index=False)
    print(f"Saved primary OOD-suffixed bin metrics: {bin_primary_suffix}")
    display(df_bin_metrics.head(16))
else:
    print("No bin-level metrics were produced.")


# Secondary OOD diagnostics (suffixed outputs)
for ood_name in OOD_SECONDARY_SPLITS:
    artifacts_secondary = SECONDARY_MODEL_ARTIFACTS.get(ood_name, [])
    if not artifacts_secondary:
        continue

    ood_prompts_sec = df_ood_map[ood_name]["prompt_text"].reset_index(drop=True)
    ood_bd_sec = ibvs2_bd_ood_map[ood_name].reset_index(drop=True)
    y_ood_sec = np.asarray(y_ood_map[ood_name], dtype=int)

    ood_masks_sec = _slice_masks(ood_prompts_sec, ood_bd_sec)

    activation_rows_sec = _activation_summary("OOD", y_ood_sec, ood_bd_sec, ood_masks_sec)
    if activation_rows_sec:
        df_activation_sec = pd.DataFrame(activation_rows_sec)
        df_activation_sec["ood_name"] = ood_name
        activation_sec_path = OUT_IBVS_DIR / f"ibvs_v2_activation_summary_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_activation_sec.to_csv(activation_sec_path, index=False)
        print(f"Saved secondary OOD activation summary ({ood_name}): {activation_sec_path}")

    # Slice metrics
    slice_rows_sec = []
    neg_mask = y_ood_sec == 0
    n_neg_total = int(neg_mask.sum())

    for artifact in artifacts_secondary:
        model_name = artifact["model"]
        eval_track = artifact["eval_track"]
        proba = np.asarray(artifact["ood_proba"], dtype=float)
        pred = np.asarray(artifact["ood_pred"], dtype=int)

        for slice_name, mask in ood_masks_sec.items():
            y_s = y_ood_sec[mask]
            p_s = pred[mask]
            s_s = proba[mask]

            n_pos = int((y_s == 1).sum())
            n_neg = int((y_s == 0).sum())

            if n_pos == 0 or n_neg == 0:
                continue

            note = (
                f"slice_eval={slice_name}; stage=OOD; eval_track={eval_track}; "
                f"n_pos={n_pos}; n_neg={n_neg}; n_neg_total_stage={n_neg_total}; ood_name={ood_name}"
            )
            res = evaluate_predictions(
                split_name=f"OOD_SLICE_{slice_name.upper()}",
                y_true=y_s,
                y_pred=p_s,
                y_score_for_metrics=s_s,
                print_report=False,
                threshold_note=note,
            )

            df_one = results_to_dataframe(model_name, [res])
            df_one["eval_track"] = eval_track
            df_one["slice_stage"] = "OOD"
            df_one["slice_name"] = slice_name
            df_one["slice_positive_count"] = n_pos
            df_one["slice_negative_count"] = n_neg
            df_one["slice_negative_total_stage"] = n_neg_total
            df_one["slice_positive_prevalence_within_stage"] = float(n_pos / max(int((y_ood_sec == 1).sum()), 1))
            df_one["split_tag"] = SPLIT_TAG
            df_one["ood_name"] = ood_name
            slice_rows_sec.append(df_one)

    if slice_rows_sec:
        df_slice_sec = pd.concat(slice_rows_sec, ignore_index=True)
        slice_sec_path = OUT_METRICS_DIR / f"metrics_ablation_slice_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_slice_sec.to_csv(slice_sec_path, index=False)
        print(f"Saved secondary OOD slice metrics ({ood_name}): {slice_sec_path}")

    # Bin metrics
    stats_sec = text_stats(ood_prompts_sec)
    stats_sec["length_bin"] = safe_qcut(stats_sec["token_count"], q=4, prefix="len")
    stats_sec["complexity_bin"] = safe_qcut(stats_sec["lexical_ttr"], q=4, prefix="complex")

    bin_rows_sec = []
    for artifact in artifacts_secondary:
        model_name = artifact["model"]
        eval_track = artifact["eval_track"]
        proba = np.asarray(artifact["ood_proba"], dtype=float)
        pred = np.asarray(artifact["ood_pred"], dtype=int)

        for bin_family in ["length_bin", "complexity_bin"]:
            for bin_label in sorted([b for b in stats_sec[bin_family].dropna().unique()]):
                mask = (stats_sec[bin_family] == bin_label).to_numpy(dtype=bool)
                y_s = y_ood_sec[mask]
                p_s = pred[mask]
                s_s = proba[mask]

                n_total = int(mask.sum())
                n_pos = int((y_s == 1).sum())
                n_neg = int((y_s == 0).sum())

                if n_total == 0 or n_pos == 0 or n_neg == 0:
                    continue

                note = (
                    f"bin_eval={bin_family}:{bin_label}; stage=OOD; eval_track={eval_track}; "
                    f"n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}; ood_name={ood_name}"
                )
                res = evaluate_predictions(
                    split_name=f"OOD_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                    y_true=y_s,
                    y_pred=p_s,
                    y_score_for_metrics=s_s,
                    print_report=False,
                    threshold_note=note,
                )

                evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))

                df_one = results_to_dataframe(model_name, [res])
                df_one["eval_track"] = eval_track
                df_one["slice_stage"] = "OOD"
                df_one["bin_family"] = bin_family
                df_one["bin_label"] = str(bin_label)
                df_one["bin_count"] = n_total
                df_one["bin_positive_count"] = n_pos
                df_one["bin_negative_count"] = n_neg
                df_one["evasion_rate"] = evasion_rate
                df_one["token_count_median"] = float(np.median(stats_sec.loc[mask, "token_count"].astype(float)))
                df_one["token_count_mean"] = float(np.mean(stats_sec.loc[mask, "token_count"].astype(float)))
                df_one["lexical_ttr_median"] = float(np.median(stats_sec.loc[mask, "lexical_ttr"].astype(float)))
                df_one["avg_token_len_median"] = float(np.median(stats_sec.loc[mask, "avg_token_len"].astype(float)))
                df_one["split_tag"] = SPLIT_TAG
                df_one["ood_name"] = ood_name
                bin_rows_sec.append(df_one)

    if bin_rows_sec:
        df_bin_sec = pd.concat(bin_rows_sec, ignore_index=True)
        bin_sec_path = OUT_METRICS_DIR / f"metrics_ablation_bins_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_bin_sec.to_csv(bin_sec_path, index=False)
        print(f"Saved secondary OOD bin metrics ({ood_name}): {bin_sec_path}")


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 12) Save IBVS breakdowns (v2 only; v1 breakdown is archival and not regenerated)
OUT_IBVS_DIR = PROJECT_ROOT / "experiments" / "results" / "ibvs"
OUT_IBVS_DIR.mkdir(parents=True, exist_ok=True)

blocks = [
    pd.concat([ibvs2_bd_train.reset_index(drop=True), ibvs2_tr_train.reset_index(drop=True)], axis=1).assign(split="train"),
    pd.concat([ibvs2_bd_val.reset_index(drop=True), ibvs2_tr_val.reset_index(drop=True)], axis=1).assign(split="val"),
    pd.concat([ibvs2_bd_test.reset_index(drop=True), ibvs2_tr_test.reset_index(drop=True)], axis=1).assign(split="test"),
    pd.concat([ibvs2_bd_ood.reset_index(drop=True), ibvs2_tr_ood.reset_index(drop=True)], axis=1).assign(split="ood_test"),
]

for ood_name in OOD_SECONDARY_SPLITS:
    bd = ibvs2_bd_ood_map.get(ood_name)
    tr = ibvs2_tr_ood_map.get(ood_name)
    if bd is None or tr is None:
        continue
    blocks.append(pd.concat([bd.reset_index(drop=True), tr.reset_index(drop=True)], axis=1).assign(split=ood_name))

ibvs2_breakdown_all = pd.concat(blocks, ignore_index=True)
ibvs2_breakdown_path = OUT_IBVS_DIR / f"ibvs_v2_breakdown_split{SPLIT_TAG}.csv"
ibvs2_breakdown_all.to_csv(ibvs2_breakdown_path, index=False)
print(f"Saved IBVS v2 breakdown: {ibvs2_breakdown_path}")


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 15) Dissertation views

def split_key(s: str) -> int:
    s = str(s)
    if "VAL" in s:
        return 0
    if "TEST" in s:
        return 1
    if "OOD" in s:
        return 2
    return 99

pretty = ablation_metrics.copy()
pretty["eval_track"] = pretty.get("eval_track", "deployment_threshold")
pretty["_split_key"] = pretty["split"].apply(split_key)
pretty = pretty.sort_values(by=["eval_track", "model", "_split_key"]).drop(columns=["_split_key"]).reset_index(drop=True)

for c in [
    "acc", "macro_f1", "auc_pr", "roc_auc",
    "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr"
]:
    if c in pretty.columns:
        pretty[c] = pretty[c].astype(float).round(4)

print("\n=== Pretty Ablation Table (all tracks) ===")
display(pretty)

ranking_ood = pretty[(pretty["eval_track"] == "ranking") & (pretty["split"] == "OOD")].copy()
ranking_ood = ranking_ood.sort_values(by=["tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "macro_f1"], ascending=False).reset_index(drop=True)
print("\n=== OOD Focus (ranking track: TPR@1%, TPR@5%, macro-F1) ===")
display(ranking_ood)

deploy_ood = pretty[(pretty["eval_track"] == "deployment_threshold") & (pretty["split"] == "OOD")].copy()
deploy_ood = deploy_ood.sort_values(by=["macro_f1", "tpr_at_5pct_fpr", "tpr_at_1pct_fpr"], ascending=False).reset_index(drop=True)
print("\n=== OOD Focus (deployment track: macro-F1, TPR@5%, TPR@1%) ===")
display(deploy_ood)


In [ ]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 16) Hard-negative extraction for interpretability (no CI / manifest / gate CSV outputs)
CHAMPION_MODEL = "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL"


def _artifact(model_name: str, eval_track: str):
    for a in MODEL_ARTIFACTS:
        if a["model"] == model_name and a["eval_track"] == eval_track:
            return a
    raise KeyError(f"Artifact not found: model={model_name}, eval_track={eval_track}")


def _artifact_for_split(ood_name: str, model_name: str, eval_track: str):
    artifacts = SECONDARY_MODEL_ARTIFACTS.get(ood_name, [])
    for a in artifacts:
        if a["model"] == model_name and a["eval_track"] == eval_track:
            return a
    raise KeyError(f"Artifact not found for {ood_name}: model={model_name}, eval_track={eval_track}")


def _threshold_for_target_fpr(y_true: np.ndarray, y_score: np.ndarray, target_fpr: float = 0.01) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    thresholds = np.unique(y_score)
    best_t = float(thresholds.max())
    best_tpr = -1.0
    best_fpr = float("inf")

    for t in thresholds:
        pred = (y_score >= t).astype(int)
        neg = y_true == 0
        pos = y_true == 1
        fpr = float(pred[neg].mean()) if neg.any() else float("nan")
        tpr = float(pred[pos].mean()) if pos.any() else float("nan")
        if np.isfinite(fpr) and fpr <= target_fpr:
            if (tpr > best_tpr) or (np.isclose(tpr, best_tpr) and fpr < best_fpr):
                best_t = float(t)
                best_tpr = float(tpr)
                best_fpr = float(fpr)

    return best_t


def _build_fp_tables(df_prompts, bd_df, tr_df, champ_rank, champ_deploy, ood_name):
    t_1pct = _threshold_for_target_fpr(champ_rank["y_ood"], champ_rank["ood_proba"], target_fpr=0.01)
    mask_fp_1pct = (champ_rank["y_ood"] == 0) & (champ_rank["ood_proba"] >= t_1pct)
    mask_fp_deploy = (champ_deploy["y_ood"] == 0) & (champ_deploy["ood_pred"] == 1)

    fp_tables = []
    for region_name, mask, score_vec, threshold_used in [
        ("ood_near_1pct_fpr", mask_fp_1pct, champ_rank["ood_proba"], t_1pct),
        ("ood_deployment_threshold", mask_fp_deploy, champ_deploy["ood_proba"], champ_deploy["t_star"]),
    ]:
        idx = np.where(mask)[0]
        if idx.size == 0:
            continue
        block = pd.concat(
            [
                df_prompts.iloc[idx].reset_index(drop=True),
                bd_df.iloc[idx].reset_index(drop=True),
                tr_df.iloc[idx].reset_index(drop=True),
            ],
            axis=1,
        )
        block["analysis_region"] = region_name
        block["score"] = np.asarray(score_vec, dtype=float)[idx]
        block["threshold_used"] = float(threshold_used)
        block["idx_ood"] = idx
        block["ood_name"] = ood_name
        fp_tables.append(block)

    return fp_tables


def _save_fp_outputs(fp_tables, *, ood_name: str, write_unsuffixed: bool = False):
    if not fp_tables:
        print(f"No OOD false positives found for {ood_name} in selected regions.")
        return

    fp_cases = pd.concat(fp_tables, ignore_index=True)
    fp_cases = fp_cases.sort_values(by=["analysis_region", "score"], ascending=[True, False]).reset_index(drop=True)

    if write_unsuffixed:
        fp_cases_path = OUT_IBVS_DIR / f"ibvs_v2_fp_cases_split{SPLIT_TAG}.csv"
        fp_cases.to_csv(fp_cases_path, index=False)
        print(f"Saved FP cases: {fp_cases_path}")

    if not WRITE_MINIMAL_OUTPUTS:
        fp_cases_suffix = OUT_IBVS_DIR / f"ibvs_v2_fp_cases_split{SPLIT_TAG}__ood-{ood_name}.csv"
        fp_cases.to_csv(fp_cases_suffix, index=False)
        print(f"Saved OOD-suffixed FP cases ({ood_name}): {fp_cases_suffix}")

    trigger_summary = (
        fp_cases.assign(trigger_rule=fp_cases["triggered_rules"].fillna("").str.split("|"))
        .explode("trigger_rule")
    )
    trigger_summary = trigger_summary[trigger_summary["trigger_rule"].astype(str).str.len() > 0]
    trigger_summary = (
        trigger_summary.groupby(["analysis_region", "trigger_rule"]).size().reset_index(name="count")
        .sort_values(by=["analysis_region", "count"], ascending=[True, False])
        .reset_index(drop=True)
    )
    trigger_summary["ood_name"] = ood_name

    if write_unsuffixed:
        trigger_summary_path = OUT_IBVS_DIR / f"ibvs_v2_fp_trigger_summary_split{SPLIT_TAG}.csv"
        trigger_summary.to_csv(trigger_summary_path, index=False)
        print(f"Saved FP trigger summary: {trigger_summary_path}")

    if not WRITE_MINIMAL_OUTPUTS:
        trigger_summary_suffix = OUT_IBVS_DIR / f"ibvs_v2_fp_trigger_summary_split{SPLIT_TAG}__ood-{ood_name}.csv"
        trigger_summary.to_csv(trigger_summary_suffix, index=False)
        print(f"Saved OOD-suffixed FP trigger summary ({ood_name}): {trigger_summary_suffix}")


# Primary OOD
champ_rank = _artifact(CHAMPION_MODEL, "ranking")
champ_deploy = _artifact(CHAMPION_MODEL, "deployment_threshold")
ood_prompts = df_ood[["prompt_text", "label"]].reset_index(drop=True)
ood_breakdown = ibvs2_bd_ood.reset_index(drop=True)
ood_triggers = ibvs2_tr_ood.reset_index(drop=True)
fp_primary = _build_fp_tables(ood_prompts, ood_breakdown, ood_triggers, champ_rank, champ_deploy, "ood_test")
_save_fp_outputs(fp_primary, ood_name="ood_test", write_unsuffixed=True)

# Secondary OOD sets
for ood_name in OOD_SECONDARY_SPLITS:
    artifacts = SECONDARY_MODEL_ARTIFACTS.get(ood_name, [])
    if not artifacts:
        continue
    try:
        champ_rank_sec = _artifact_for_split(ood_name, CHAMPION_MODEL, "ranking")
        champ_deploy_sec = _artifact_for_split(ood_name, CHAMPION_MODEL, "deployment_threshold")
    except KeyError as exc:
        print(f"Skipping FP extraction for {ood_name}: {exc}")
        continue

    ood_prompts_sec = df_ood_map[ood_name][["prompt_text", "label"]].reset_index(drop=True)
    ood_breakdown_sec = ibvs2_bd_ood_map[ood_name].reset_index(drop=True)
    ood_triggers_sec = ibvs2_tr_ood_map[ood_name].reset_index(drop=True)
    fp_secondary = _build_fp_tables(ood_prompts_sec, ood_breakdown_sec, ood_triggers_sec, champ_rank_sec, champ_deploy_sec, ood_name)
    _save_fp_outputs(fp_secondary, ood_name=ood_name, write_unsuffixed=False)


<!-- NOTEBOOK_OUTPUT_SUMMARY -->
## 6. Output Summary
1. Writes canonical ablation metrics for the active split to `experiments/results/metrics/metrics_ablation_split{tag}.csv`.
2. When full-output mode is enabled, also writes OOD-specific suffixed metrics, slice diagnostics, and difficulty-bin tables.
3. Writes IBVS forensic artifacts including:
   - `ibvs_v2_breakdown_split{tag}.csv`
   - `ibvs_v2_activation_summary_split{tag}.csv`
   - `ibvs_v2_fp_cases_split{tag}.csv`
   - `ibvs_v2_fp_trigger_summary_split{tag}.csv`
